In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol

## Data Preprocessing and Training Machine Learning Models

- Calculate basic molecular properties of identified target-binding ligands using RDKit
- Additional data preparation (scaling)
- Train machine learning models, e.g.:
    - Multiple Linear Regression (MLR)
    - Ridge Regression
    - Support Vector Machine (SVM)
    - Decision Tree
    - Random Forest (RF)
    - Gradient Boosting Machine (GBM)
    - Feedforward Neural Network (FFNN)
    - Graph Neural Network (GNN)

### Reading in Data

In [ ]:
chembl_id = "CHEMBL402"

df = pd.read_csv(f"../data/{chembl_id}.csv").drop(columns=["Unnamed: 0"])
df

In [ ]:
# Add mol-type objects to dataframe for calculating molecular properties (see below)

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllColumns = True

# Add mol-type objects without H atoms
df["ROMol"] = df["canonical_smiles"].apply(Chem.MolFromSmiles)
# Add mol-type objects including H atoms
df["ROMolH"] = df["ROMol"].apply(Chem.AddHs)
# Generate 3D coordinates
df["ROMol3D"] = df.apply(lambda row: AllChem.EmbedMolecule(row["ROMolH"], randomSeed=42), axis=1)
df = df[df["ROMol3D"] == 0].reset_index().drop(columns=["index"])
# Optimize the 3D structure using the MMFF94 force field
df["ROMolOPT"] = df.apply(lambda row: AllChem.MMFFOptimizeMolecule(row["ROMolH"], maxIters=2000), axis=1)
df = df[df["ROMolOPT"] == 0].reset_index().drop(columns=["index"])
df

In [ ]:
def display_molecule_3D(mol):
    """
    Display a 3D molecule using py3Dmol.
        Requires a molecule with 3D coordinates.
        mol: RDKit Mol object with 3D coordinates
    """
    mb = Chem.MolToMolBlock(mol)
    p = py3Dmol.view(width=400, height=400)
    p.addModel(mb, 'sdf')
    p.setStyle({'stick':{'colorscheme':'greyCarbon', 'radius': 0.1}, 'sphere':{'scale':0.25}})
    p.setBackgroundColor('0xeeeeee')
    p.zoomTo()
    display(p.show())

In [ ]:
display_molecule_3D(df["ROMolH"][0])

### Calculating Molecular Properties

In [ ]:
# Calculate basic molecular properties of identified target-binding ligands using RDKit

from rdkit.Chem import Descriptors, Descriptors3D

df["MW"] = df["ROMol"].apply(Descriptors.MolWt)
df["logP"] = df["ROMol"].apply(Descriptors.MolLogP)
df["HBA"] = df["ROMol"].apply(Descriptors.NumHAcceptors)
df["HBD"] = df["ROMol"].apply(Descriptors.NumHDonors)

def Ro5(mw, logP, HBA, HBD) -> int:
    Ro5_counter = 0

    if mw <= 500:
      Ro5_counter += 1
    if logP <= 5:
      Ro5_counter += 1
    if HBA <= 10:
      Ro5_counter += 1
    if HBD <= 5:
      Ro5_counter += 1

    return Ro5_counter

df["Ro5"] = df.apply(lambda row: Ro5(row["MW"], row["logP"], row["HBA"], row["HBD"]), axis=1)
df["RB"] = df["ROMol"].apply(Descriptors.NumRotatableBonds)
df["TPSA"] = df["ROMol"].apply(Descriptors.TPSA)
df["ROG"] = df.apply(lambda row: Descriptors3D.RadiusOfGyration(row["ROMolH"]), axis=1)
df

In [ ]:
# Sanity check!
df.describe()

### Splitting into Training, Validation, and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split
trainval_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(trainval_df, test_size=0.25, random_state=42)

train_df = train_df.reset_index().drop(columns=["index"])
val_df = val_df.reset_index().drop(columns=["index"])
test_df = test_df.reset_index().drop(columns=["index"])

In [ ]:
input_cols = ["MW", "logP", "HBA", "HBD", "Ro5", "RB", "TPSA", "ROG"]
target_col = "pIC50"

In [ ]:
train_inputs = train_df[input_cols].copy()
val_inputs = val_df[input_cols].copy()
test_inputs = test_df[input_cols].copy()

train_targets = train_df[target_col].copy()
val_targets = val_df[target_col].copy()
test_targets = test_df[target_col].copy()

### Additional Data Preparation (Scaling)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(df[input_cols])

In [ ]:
train_inputs[input_cols] = scaler.transform(train_inputs[input_cols])
val_inputs[input_cols] = scaler.transform(val_inputs[input_cols])
test_inputs[input_cols] = scaler.transform(test_inputs[input_cols])

In [ ]:
train_inputs.describe().loc[['min', 'max']]

In [ ]:
val_inputs.describe().loc[['min', 'max']]

In [ ]:
test_inputs.describe().loc[['min', 'max']]

### Data Preprocessing Summary

Several dataframes have been created to now hold (scaled) input molecular descriptors and target pIC50 values:
- Input dataframes: `train_inputs`, `val_inputs`, and `test_inputs`
- Target dataframes: `train_targets`, `val_targets`, and `test_targets`

## Baseline Models: Primitive Predictive Models

### Always Predicting the Mean of the Training pIC50 Values

In [ ]:
def mean_guess(inputs, targets):
    return np.full(inputs.shape[0], fill_value=targets.mean())

In [ ]:
train_preds = mean_guess(train_inputs, train_targets)
val_preds = mean_guess(val_inputs, train_targets)        # Use training target range for validation predictions!   
test_preds = mean_guess(test_inputs, train_targets)      # Use training target range for test predictions!

In [ ]:
# Evaluation of "Always Predict the Mean" Model

from sklearn.metrics import root_mean_squared_error

train_rmse = root_mean_squared_error(train_targets, train_preds)
val_rmse = root_mean_squared_error(val_targets, val_preds)
test_rmse = root_mean_squared_error(test_targets, test_preds)

print("Baseline Model Performance (Always Predicting the Mean):")
print(f"Train RMSE (pIC50):         {train_rmse:.4f}")
print(f"Validation RMSE (pIC50):    {val_rmse:.4f}")
print(f"Test RMSE (pIC50):          {test_rmse:.4f}")

### Predicting Random Numbers in the Range of the Training pIC50 Values

In [ ]:
np.random.seed(42)  # For reproducibility

def random_guess(inputs, targets):
    return np.random.uniform(targets.min(), targets.max(), len(inputs))

In [ ]:
train_preds = random_guess(train_inputs, train_targets)
val_preds = random_guess(val_inputs, train_targets)         # Use training target range for validation predictions!
test_preds = random_guess(test_inputs, train_targets)       # Use training target range for test predictions!

In [ ]:
# Evaluation of "Predict Random Numbers" Model

from sklearn.metrics import root_mean_squared_error

train_rmse = root_mean_squared_error(train_targets, train_preds)
val_rmse = root_mean_squared_error(val_targets, val_preds)
test_rmse = root_mean_squared_error(test_targets, test_preds)

print("Baseline Model Performance (Predicting Random Numbers):")
print(f"Train RMSE (pIC50):         {train_rmse:.4f}")
print(f"Validation RMSE (pIC50):    {val_rmse:.4f}")
print(f"Test RMSE (pIC50):          {test_rmse:.4f}")

## Classical Machine Learning Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [ ]:
model_name_list = []
test_rmse_list = []

In [ ]:
def model_train_val_evaluate(model, model_name):
    model.fit(train_inputs, train_targets)
    train_preds = model.predict(train_inputs)
    val_preds = model.predict(val_inputs)

    train_rmse = root_mean_squared_error(train_targets, train_preds)
    val_rmse = root_mean_squared_error(val_targets, val_preds)

    print(f"{model_name} Model Performance:")
    print(f"Train RMSE (pIC50):         {train_rmse:.4f}")
    print(f"Validation RMSE (pIC50):    {val_rmse:.4f}")

    return model


def model_test_evaluate(model, model_name, model_name_list=model_name_list, test_rmse_list=test_rmse_list):
    test_preds = model.predict(test_inputs)
    test_rmse = root_mean_squared_error(test_targets, test_preds)
    model_name_list.append(model_name)
    test_rmse_list.append(test_rmse)
    print(f"\t{model_name} Test Performance:\n\t {test_rmse:.4f}")

    return None

In [ ]:
model = model_train_val_evaluate(LinearRegression(), "MLR")
model_test_evaluate(model, "MLR")

In [ ]:
weights_df = pd.DataFrame({
    'feature': np.append(input_cols, 'intercept'),
    'weight': np.append(model.coef_, model.intercept_)
}).sort_values('weight', ascending=False)

plt.bar(x=weights_df['feature'], height=weights_df['weight'], color="navy")
plt.xlabel("Feature")
plt.ylabel("Weight / $-$")
plt.title("Multiple Linear Regression (MLR)")
plt.show()

In [ ]:
model = model_train_val_evaluate(Ridge(random_state=42), "Ridge")
model_test_evaluate(model, "Ridge")

In [ ]:
weights_df = pd.DataFrame({
    'feature': np.append(input_cols, 'intercept'),
    'weight': np.append(model.coef_, model.intercept_)
}).sort_values('weight', ascending=False)

plt.bar(x=weights_df['feature'], height=weights_df['weight'], color="navy")
plt.xlabel("Feature")
plt.ylabel("Weight / $-$")
plt.title("Ridge Regression")
plt.show()

In [ ]:
train_rmse_list = []
val_rmse_list = []
for i in np.arange(0.01, 0.5, 0.025):
    model = SVR(C=2, epsilon=i)
    model.fit(train_inputs, train_targets)
    train_preds = model.predict(train_inputs)
    val_preds = model.predict(val_inputs)

    train_rmse = root_mean_squared_error(train_targets, train_preds)
    val_rmse = root_mean_squared_error(val_targets, val_preds)
    train_rmse_list.append(train_rmse)
    val_rmse_list.append(val_rmse)

plt.plot(np.arange(0.01, 0.5, 0.025), train_rmse_list, label='Train RMSE', color='maroon')
plt.plot(np.arange(0.01, 0.5, 0.025), val_rmse_list, label='Validation RMSE', color='navy')
plt.xlabel('Epsilon / $-$')
plt.ylabel('RMSE (pIC50) / $-$')
plt.title('SVR Performance')
plt.legend()
plt.show()

In [ ]:
model = model_train_val_evaluate(SVR(C=2, epsilon=0.15), "SVR")
model_test_evaluate(model, "SVR")

In [ ]:
train_rmse_list = []
val_rmse_list = []
for i in range(3, 10):
    model = DecisionTreeRegressor(max_depth=i, random_state=42)
    model.fit(train_inputs, train_targets)
    train_preds = model.predict(train_inputs)
    val_preds = model.predict(val_inputs)

    train_rmse = root_mean_squared_error(train_targets, train_preds)
    val_rmse = root_mean_squared_error(val_targets, val_preds)
    train_rmse_list.append(train_rmse)
    val_rmse_list.append(val_rmse)

plt.plot(range(3, 10), train_rmse_list, label='Train RMSE', color='maroon')
plt.plot(range(3, 10), val_rmse_list, label='Validation RMSE', color='navy')
plt.xlabel('Max. Tree Depth / $-$')
plt.ylabel('RMSE (pIC50) / $-$')
plt.title('Decision Tree Regressor Performance')
plt.legend()
plt.show()

In [ ]:
model = model_train_val_evaluate(DecisionTreeRegressor(max_depth=6, random_state=42), "Decision Tree")
model_test_evaluate(model, "Decision Tree")

In [ ]:
importance_df = pd.DataFrame({
    'feature': input_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.bar(x=importance_df['feature'], height=importance_df['importance'], color="navy")
plt.xlabel("Feature")
plt.ylabel("Importance / $-$")
plt.title("Decision Tree Regressor")
plt.show()

In [ ]:
train_rmse_list = []
val_rmse_list = []
for i in range(2, 10):
    model = RandomForestRegressor(n_estimators=5, max_depth=i, random_state=42, n_jobs=-1)
    model.fit(train_inputs, train_targets)
    train_preds = model.predict(train_inputs)
    val_preds = model.predict(val_inputs)

    train_rmse = root_mean_squared_error(train_targets, train_preds)
    val_rmse = root_mean_squared_error(val_targets, val_preds)
    train_rmse_list.append(train_rmse)
    val_rmse_list.append(val_rmse)

plt.plot(range(2, 10), train_rmse_list, label='Train RMSE', color='maroon')
plt.plot(range(2, 10), val_rmse_list, label='Validation RMSE', color='navy')
plt.xlabel('Max. Tree Depth / $-$')
plt.ylabel('RMSE (pIC50) / $-$')
plt.title('Random Forest Regressor Performance')
plt.legend()
plt.show()

In [ ]:
model = model_train_val_evaluate(RandomForestRegressor(n_estimators=5, max_depth=4, random_state=42, n_jobs=-1), "RF")
model_test_evaluate(model, "RF")

In [ ]:
importance_df = pd.DataFrame({
    'feature': input_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.bar(x=importance_df['feature'], height=importance_df['importance'], color="navy")
plt.xlabel("Feature")
plt.ylabel("Importance / $-$")
plt.title("Random Forest Regressor")
plt.show()

In [ ]:
train_rmse_list = []
val_rmse_list = []
for i in range(2, 25):
    model = XGBRegressor(n_estimators=i, max_depth=3, learning_rate=0.2, random_state=42, n_jobs=-1)
    model.fit(train_inputs, train_targets)
    train_preds = model.predict(train_inputs)
    val_preds = model.predict(val_inputs)

    train_rmse = root_mean_squared_error(train_targets, train_preds)
    val_rmse = root_mean_squared_error(val_targets, val_preds)
    train_rmse_list.append(train_rmse)
    val_rmse_list.append(val_rmse)

plt.plot(range(2, 25), train_rmse_list, label='Train RMSE', color='maroon')
plt.plot(range(2, 25), val_rmse_list, label='Validation RMSE', color='navy')
plt.xlabel('Number of Estimators / $-$')
plt.ylabel('RMSE (pIC50) / $-$')
plt.title('XGBoost Performance')
plt.legend()
plt.show()

In [ ]:
model = model_train_val_evaluate(XGBRegressor(n_estimators=15, max_depth=3, learning_rate=0.2, random_state=42, n_jobs=-1), "XGBoost")
model_test_evaluate(model, "XGBoost")

In [ ]:
importance_df = pd.DataFrame({
    'feature': input_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.bar(x=importance_df['feature'], height=importance_df['importance'], color="navy")
plt.xlabel("Feature")
plt.ylabel("Importance / $-$")
plt.title("XGBoost")
plt.savefig("../data/xgboost_feature_importance.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

### Model Overiew $-$ RMSE Values of Test Set

In [ ]:
plt.bar(x=model_name_list, height=test_rmse_list, color="green")
plt.ylabel("Test Set RMSE (pIC50) / $-$")
plt.title(f"Model Performance Comparison $-$ ID {chembl_id}")
plt.savefig("../data/model_performance_comparison.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

### Qualitative Observation

All tree-based methods (Decision Tree, RF, XGBoost) unanimously highlight the importance of the 2D **TPSA** as a feature!

## Deep Learning Models

Inspirations taken from volkamerlab on GitHub (Prof. Andrea Volkamer, Saarbrücken/Germany)
- https://github.com/volkamerlab/cic_summerschool_2025 
- https://github.com/volkamerlab/ai_in_chemistry_workshop

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch import optim

### Feed Forward Neural Network (FFNN)

Idea: Use the 3D molecular structures (from the data above) to create Morgan fingerprints which can be used as input for the FFNN. The output will be the pIC50 value of the input molecule.

_Future idea: Try the above defined molecular descriptors (e.g., donors/acceptors, TPSA) as input instead_

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

morganFPgen = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)

def molToMorganFP(molobject):
    return morganFPgen.GetFingerprintAsNumPy(molobject)

class ActivityFFNNDataset(Dataset):
    def __init__(self, mol3D, pIC50):
        self.morganFP = torch.tensor(
            list(map(molToMorganFP, mol3D)), dtype=torch.float
        )
        self.pIC50 = torch.tensor(pIC50, dtype=torch.float)

    def __len__(self):
        return len(self.pIC50)

    def __getitem__(self, idx):
        morganFP = self.morganFP[idx]
        pIC50 = self.pIC50[idx]
        return morganFP, pIC50

### FFNN/1: Data Preparation

In [ ]:
train_dataset = ActivityFFNNDataset(train_df["ROMolH"], train_df["pIC50"])
val_dataset = ActivityFFNNDataset(val_df["ROMolH"], val_df["pIC50"])
test_dataset = ActivityFFNNDataset(test_df["ROMolH"], test_df["pIC50"])

In [ ]:
# Test the methods of the custom ActivityFFNNDataset class
print(len(train_dataset), len(val_dataset), len(test_dataset))
print(train_dataset[0], val_dataset[0], test_dataset[0])

In [ ]:
torch.manual_seed(42)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### FFNN/2: Model Definition

In [ ]:
import torch.nn as nn

class ActivityFFNNNet(nn.Module):
    def __init__(self):
        super(ActivityFFNNNet, self).__init__()
        # Input layer (2048 neurons) corresponds to the MorganFP fpSize of 2048!
        self.fc1 = nn.Linear(2048, 1024)
        self.fc2 = nn.Linear(1024, 1)
        self.relu = nn.ReLU()

    # Define the forward pass (i.e., architecture of the FFNN)
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
from torchsummary import summary
activityFFNNmodel = ActivityFFNNNet()
summary(activityFFNNmodel, (2048,));    # Semicolon prevents redundant (duplicate) output here

### FFNN/3: Model Training ("Early Stopping")

In [ ]:
torch.manual_seed(42)
model = ActivityFFNNNet()

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# MSE loss function
criterion = nn.MSELoss()

def train_step(loader_train, model=model, criterion=criterion, optimizer=optimizer):
    model.train()
    total_loss_train = 0

    for morganFP, pIC50 in loader_train:
        
        optimizer.zero_grad()                               # reset the gradients after every iteration
        prediction = model(morganFP).squeeze()              # forward step, squeeze to transform a (N,1) tensor (matrix) to (N,) tensor (vector)
        batch_loss = criterion(prediction, pIC50)           # loss for current batch
        batch_loss.backward()                               # backpropagation
        optimizer.step()                                    # updates parameters (weights & bias)

        total_loss_train += batch_loss.item() * len(pIC50)

    return total_loss_train / len(loader_train)

def val_step(loader_val, model=model, criterion=criterion): # Notice: NO optimizer here!
    model.eval()
    total_loss_val = 0.0

    # do not change the gradients
    with torch.no_grad():
        for morganFP, pIC50 in loader_val:
            prediction = model(morganFP).squeeze()
            total_loss_val += criterion(prediction, pIC50).item()

    return total_loss_val / len(loader_val)

def test_step(loader_test, model=model, criterion=criterion): # identical to val_step() but for clarity
    model.eval()
    total_loss_test = 0.0

    # do not change the gradients
    with torch.no_grad():
        for morganFP, pIC50 in loader_test:
            prediction = model(morganFP).squeeze()
            total_loss_test += criterion(prediction, pIC50).item()

    return total_loss_test / len(loader_test)

In [ ]:
train_losses, val_losses = [], []

for epoch in range(100):
    train_loss = train_step(train_dataloader, model, criterion, optimizer)
    valid_loss = val_step(val_dataloader, model, criterion)

    train_losses.append(train_loss)
    val_losses.append(valid_loss)

    print(f"Epoch: {epoch + 1:03d}, Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}")

In [ ]:
plt.plot([np.sqrt(mse) for mse in train_losses], label='Train RMSE', color='maroon')
plt.plot([np.sqrt(mse) for mse in val_losses], label='Validation RMSE', color='navy')
plt.title('Overtraining Analysis of FFNN Model')
plt.xlabel('Epochs / $-$')
plt.ylabel('RMSE (pIC50) / $-$')
plt.ylim(0, 5)
plt.legend()
plt.show()

In [ ]:
# RESET EVERYTHING - i.e., new model

torch.manual_seed(42)
model_opt = ActivityFFNNNet()
optimizer = optim.Adam(model_opt.parameters(), lr=1e-3)
criterion = nn.MSELoss()

train_losses, val_losses = [], []

num_epochs = 20
for epoch in range(num_epochs): # 20 epochs only to prevent overfitting ("early stopping"), see above
    train_loss = train_step(train_dataloader, model_opt, criterion, optimizer)
    valid_loss = val_step(val_dataloader, model_opt, criterion)

    train_losses.append(train_loss)
    val_losses.append(valid_loss)

    print(f"Epoch: {epoch + 1:03d}, Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}")

In [ ]:
print(f"After {num_epochs} epochs: Training Loss (RMSE): {np.sqrt(train_loss):.4f}, Validation Loss (RMSE): {np.sqrt(valid_loss):.4f}")

In [ ]:
test_loss = test_step(test_dataloader, model_opt, criterion)
print(f"Test Loss (RMSE): {np.sqrt(test_loss):.4f}")

### Conclusion of FFNN

The simple FFNN with one hidden layer, based on Morgan fingerprints compares very well to the best tree-based classical machine learning methods (e.g., XGboost, Random Forest).

### Graph Neural Network (GNN)

In [ ]:
import torch_geometric

Idea: Use the 3D molecular structures (from the data above), i.e., coordinates, as input for the GNN. The output will be the pIC50 value of the input molecule.